In [ ]:
# ============================================================
# CELL 1: Verify GPU is available
#
# WHY: YOLOv8 training requires a CUDA-compatible GPU.
#      Without GPU, 50-epoch training would take 40-80 hours
#      on CPU, making this verification a hard prerequisite.
#      This cell also validates PyTorch version compatibility
#      with the installed CUDA driver.
# ============================================================

import torch
import os

print("=" * 55)
print(f"PyTorch Version  : {torch.__version__}")
print(f"CUDA Available   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"Number of GPUs   : {num_gpus}")
    for i in range(num_gpus):
        name = torch.cuda.get_device_name(i)
        mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}          : {name}  ({mem:.2f} GB VRAM)")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change Runtime Type > GPU")
    raise SystemExit("GPU required — stopping execution.")

print("=" * 55)
print("GPU verification passed. Proceeding with pipeline.")

In [ ]:
# ============================================================
# CELL 2: Install all required libraries
#
# WHY: Ultralytics provides the YOLOv8 framework.
#      pycocotools lets us parse COCO JSON annotations.
#      We pin ultralytics at v8.2.0 to prevent breaking
#      changes and ensure identical results across sessions.
# ============================================================

!pip install ultralytics==8.2.0 -q
!pip install pycocotools -q

# Verify installation
from ultralytics import YOLO
import ultralytics
print(f"Ultralytics version : {ultralytics.__version__}")

from pycocotools.coco import COCO
print("pycocotools loaded successfully")

print("\nAll required libraries installed and verified.")

In [ ]:
# ============================================================
# CELL 3: Download COCO 2017 from official servers
#
# WHY: cocodataset.org provides public HTTP access — no login,
#      no API key required. We download only the three
#      necessary archives:
#        - annotations_trainval2017.zip  (~241 MB)
#        - val2017.zip                   (~1 GB,  5K images)
#        - train2017.zip                 (~18 GB, 118K images)
#
#      Archives are extracted and deleted to reclaim disk space.
#      Final directory structure:
#        /content/coco/images/train2017/
#        /content/coco/images/val2017/
#        /content/coco/annotations/
# ============================================================

import os

# Create base directory structure
os.makedirs("/content/coco/images", exist_ok=True)
os.makedirs("/content/coco/annotations", exist_ok=True)

print("Downloading COCO 2017 annotations (~241 MB)...")
!wget -q --show-progress -O /content/coco/annotations.zip \
    http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q /content/coco/annotations.zip -d /content/coco/
!rm /content/coco/annotations.zip
print("Annotations downloaded and extracted.")

print("\nDownloading COCO 2017 val images (~1 GB)...")
!wget -q --show-progress -O /content/coco/images/val2017.zip \
    http://images.cocodataset.org/zips/val2017.zip
!unzip -q /content/coco/images/val2017.zip -d /content/coco/images/
!rm /content/coco/images/val2017.zip
print("Val images downloaded and extracted.")

print("\nDownloading COCO 2017 train images (~18 GB — this takes ~15 mins)...")
!wget -q --show-progress -O /content/coco/images/train2017.zip \
    http://images.cocodataset.org/zips/train2017.zip
!unzip -q /content/coco/images/train2017.zip -d /content/coco/images/
!rm /content/coco/images/train2017.zip
print("Train images downloaded and extracted.")

# Verify counts
train_count = len(os.listdir("/content/coco/images/train2017"))
val_count   = len(os.listdir("/content/coco/images/val2017"))
print(f"\nTrain images : {train_count:,}")
print(f"Val images   : {val_count:,}")
print("Dataset download complete.")

In [ ]:
# ============================================================
# CELL 4: Filter 15 classes from COCO and convert to YOLO format
#
# WHY: COCO has 80 classes. We select 15 everyday categories
#      that are well-represented and cover diverse real-world
#      scenarios (people, vehicles, animals, household objects).
#
# FORMAT CONVERSION:
#   COCO format : [x_min, y_min, width, height]  (absolute pixels)
#   YOLO format : [class_id, x_center, y_center, w, h]  (normalized)
#
#   x_center = (x_min + width/2)  / image_width
#   y_center = (y_min + height/2) / image_height
#   w_norm   = width  / image_width
#   h_norm   = height / image_height
#
#   All values are clamped to [0, 1] to handle COCO annotations
#   that extend to or beyond image boundaries.
#   Degenerate boxes (w_norm <= 0 or h_norm <= 0) are discarded.
# ============================================================

import json, os, shutil
from pycocotools.coco import COCO
from PIL import Image
from tqdm import tqdm

# ── 1. Define the 15 selected classes ──────────────────────────────────────
SELECTED_CLASSES = [
    'person', 'car', 'bicycle', 'motorcycle', 'bus',
    'truck', 'traffic light', 'stop sign',
    'cat', 'dog', 'bird',
    'chair', 'bottle', 'laptop', 'cell phone'
]

# ── 2. Create YOLO folder structure ────────────────────────────────────────
BASE_DIR = "/content/dataset"
for split in ['train', 'val', 'test']:
    os.makedirs(f"{BASE_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{BASE_DIR}/labels/{split}", exist_ok=True)
print("Folder structure created.")

# ── 3. Core conversion function ────────────────────────────────────────────
def convert_coco_to_yolo(ann_file, img_src_dir, out_img_dir,
                          out_lbl_dir, class_map, max_images=None):
    """
    Reads a COCO JSON annotation file, filters the 15 selected classes,
    converts bounding boxes from COCO absolute format to YOLO normalized
    format, copies images to the output directory, and writes .txt label
    files. Images are copied (not moved) to preserve the original download.
    """
    coco = COCO(ann_file)

    # Get COCO category IDs for selected classes
    selected_cat_ids = []
    for cls in SELECTED_CLASSES:
        cats = coco.getCatIds(catNms=[cls])
        selected_cat_ids.extend(cats)

    # Get all image IDs containing at least one selected class
    img_ids = set()
    for cat_id in selected_cat_ids:
        img_ids.update(coco.getImgIds(catIds=[cat_id]))
    img_ids = list(img_ids)

    # Cap image count if specified
    if max_images:
        img_ids = img_ids[:max_images]

    print(f"  Images with selected classes: {len(img_ids):,}")

    skipped = 0
    for img_id in tqdm(img_ids, desc="Converting"):
        img_info  = coco.loadImgs(img_id)[0]
        file_name = img_info['file_name']
        img_w     = img_info['width']
        img_h     = img_info['height']

        src_path = os.path.join(img_src_dir, file_name)
        if not os.path.exists(src_path):
            skipped += 1
            continue

        # Get all annotations for this image (selected classes only)
        ann_ids = coco.getAnnIds(imgIds=img_id, catIds=selected_cat_ids, iscrowd=False)
        anns    = coco.loadAnns(ann_ids)

        if len(anns) == 0:
            skipped += 1
            continue

        # Convert each annotation to YOLO normalized format
        label_lines = []
        for ann in anns:
            cat_id = ann['category_id']
            if cat_id not in class_map:
                continue

            yolo_class           = class_map[cat_id]   # remap to 0-indexed YOLO class
            x_min, y_min, w, h   = ann['bbox']

            # COCO → YOLO coordinate conversion
            x_center = (x_min + w / 2) / img_w
            y_center = (y_min + h / 2) / img_h
            w_norm   = w / img_w
            h_norm   = h / img_h

            # Clamp all values to [0, 1]
            x_center = max(0.0, min(1.0, x_center))
            y_center = max(0.0, min(1.0, y_center))
            w_norm   = max(0.0, min(1.0, w_norm))
            h_norm   = max(0.0, min(1.0, h_norm))

            # Skip degenerate bounding boxes
            if w_norm <= 0 or h_norm <= 0:
                continue

            label_lines.append(
                f"{yolo_class} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}"
            )

        if not label_lines:
            skipped += 1
            continue

        # Copy image to YOLO dataset folder
        shutil.copy2(src_path, os.path.join(out_img_dir, file_name))

        # Write YOLO label file (.txt, same stem as image)
        lbl_name = os.path.splitext(file_name)[0] + ".txt"
        with open(os.path.join(out_lbl_dir, lbl_name), 'w') as f:
            f.write("\n".join(label_lines))

    print(f"  Skipped (missing / empty): {skipped}")
    return len(img_ids) - skipped


# ── 4. Build class_map: COCO category_id → YOLO index (0–14) ───────────────
coco_tmp  = COCO("/content/coco/annotations/instances_train2017.json")
class_map = {}
for idx, cls_name in enumerate(SELECTED_CLASSES):
    for cat_id in coco_tmp.getCatIds(catNms=[cls_name]):
        class_map[cat_id] = idx

print(f"\nClass mapping built: {len(class_map)} COCO IDs → {len(SELECTED_CLASSES)} YOLO classes")
print("\nClass Index Map:")
for idx, name in enumerate(SELECTED_CLASSES):
    print(f"  {idx:2d} → {name}")

In [ ]:
# ============================================================
# CELL 5: Execute conversion and create Train / Val / Test splits
#
# SPLIT RATIOS:
#   Train (70%) → Model learns from this
#   Val   (20%) → Hyperparameter tuning, checkpoint selection
#   Test  (10%) → Final unbiased evaluation (never seen during training)
#
# WHY MAX 25,000 TRAIN IMAGES:
#   Full COCO train = 118K images (~80 GB after extraction).
#   25K images provides strong training signal while keeping
#   total disk usage manageable.
#
# TEST SPLIT STRATEGY:
#   Carved out from the original COCO val split (5,000 images)
#   by randomly selecting 20% of processed val images after
#   shuffling with random.seed(42). This ensures test images
#   are never seen during training or val-based checkpoint
#   selection.
# ============================================================

import random

print("Converting TRAIN split (up to 25,000 images)...")
n_train = convert_coco_to_yolo(
    ann_file    = "/content/coco/annotations/instances_train2017.json",
    img_src_dir = "/content/coco/images/train2017",
    out_img_dir = f"{BASE_DIR}/images/train",
    out_lbl_dir = f"{BASE_DIR}/labels/train",
    class_map   = class_map,
    max_images  = 25000
)

print("\nConverting VAL split (up to 5,000 images)...")
n_val = convert_coco_to_yolo(
    ann_file    = "/content/coco/annotations/instances_val2017.json",
    img_src_dir = "/content/coco/images/val2017",
    out_img_dir = f"{BASE_DIR}/images/val",
    out_lbl_dir = f"{BASE_DIR}/labels/val",
    class_map   = class_map,
    max_images  = 5000
)

# ── Carve out TEST split: 20% of val images (~10% of total) ────────────────
print("\nCreating TEST split from val (20% of val = ~10% of total)...")
val_images = os.listdir(f"{BASE_DIR}/images/val")
random.seed(42)          # Fixed seed for reproducibility
random.shuffle(val_images)
test_images = val_images[: len(val_images) // 5]

moved = 0
for img_file in tqdm(test_images, desc="Moving to test"):
    stem    = os.path.splitext(img_file)[0]
    src_img = f"{BASE_DIR}/images/val/{img_file}"
    src_lbl = f"{BASE_DIR}/labels/val/{stem}.txt"
    dst_img = f"{BASE_DIR}/images/test/{img_file}"
    dst_lbl = f"{BASE_DIR}/labels/test/{stem}.txt"

    if os.path.exists(src_img) and os.path.exists(src_lbl):
        shutil.move(src_img, dst_img)
        shutil.move(src_lbl, dst_lbl)
        moved += 1

# Final counts
n_train_final = len(os.listdir(f"{BASE_DIR}/images/train"))
n_val_final   = len(os.listdir(f"{BASE_DIR}/images/val"))
n_test_final  = len(os.listdir(f"{BASE_DIR}/images/test"))
n_total       = n_train_final + n_val_final + n_test_final

print(f"\n{'='*45}")
print(f"Dataset Split Summary")
print(f"{'='*45}")
print(f"  Train images : {n_train_final:,}  (~{100*n_train_final//n_total}%)")
print(f"  Val   images : {n_val_final:,}   (~{100*n_val_final//n_total}%)")
print(f"  Test  images : {n_test_final:,}   (~{100*n_test_final//n_total}%)")
print(f"  Total        : {n_total:,}")
print(f"  Random seed  : 42")
print(f"{'='*45}")

In [ ]:
# ============================================================
# CELL 6: Data Cleaning
#
# WHY THIS STEP IS CRITICAL:
#   - Corrupt images cause NaN losses and can crash training
#   - Images without labels waste compute and confuse the model
#   - Empty label files contain no target-class annotations
#   - Orphan label files (no matching image) cause errors
#   - This step ensures strict 1-to-1 image ↔ label mapping
#
# FOUR CLEANING PASSES PER SPLIT:
#   1. PIL.verify() — detect and remove corrupt images
#   2. Missing labels  — remove images with no label file
#   3. Empty labels    — remove images with blank label file
#   4. Orphan labels   — remove label files with no matching image
# ============================================================

from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

def clean_split(split_name):
    img_dir = f"{BASE_DIR}/images/{split_name}"
    lbl_dir = f"{BASE_DIR}/labels/{split_name}"

    corrupt     = 0
    no_label    = 0
    empty_label = 0
    orphan_lbl  = 0

    # Pass 1–3: Inspect every image ──────────────────────────────────────────
    for img_file in tqdm(os.listdir(img_dir), desc=f"Cleaning {split_name}"):
        img_path = os.path.join(img_dir, img_file)
        stem     = os.path.splitext(img_file)[0]
        lbl_path = os.path.join(lbl_dir, stem + ".txt")

        # Pass 1: Remove corrupt images
        try:
            with Image.open(img_path) as img:
                img.verify()
        except (UnidentifiedImageError, Exception):
            os.remove(img_path)
            if os.path.exists(lbl_path):
                os.remove(lbl_path)
            corrupt += 1
            continue

        # Pass 2: Remove images with no corresponding label file
        if not os.path.exists(lbl_path):
            os.remove(img_path)
            no_label += 1
            continue

        # Pass 3: Remove images with empty (blank) label files
        with open(lbl_path, 'r') as f:
            content = f.read().strip()
        if not content:
            os.remove(img_path)
            os.remove(lbl_path)
            empty_label += 1
            continue

    # Pass 4: Remove orphan label files (label exists but image was deleted) ─
    img_stems = {os.path.splitext(f)[0] for f in os.listdir(img_dir)}
    for lbl_file in os.listdir(lbl_dir):
        if os.path.splitext(lbl_file)[0] not in img_stems:
            os.remove(os.path.join(lbl_dir, lbl_file))
            orphan_lbl += 1

    print(f"\n  [{split_name}] Cleaning Summary:")
    print(f"    Corrupt images removed  : {corrupt}")
    print(f"    Missing labels removed  : {no_label}")
    print(f"    Empty labels removed    : {empty_label}")
    print(f"    Orphan labels removed   : {orphan_lbl}")
    print(f"    Final clean images      : {len(os.listdir(img_dir)):,}")


for split in ['train', 'val', 'test']:
    clean_split(split)

print("\nData cleaning complete — dataset is safe for training.")

In [ ]:
# ============================================================
# CELL 7: Analyze class distribution
#
# WHY: Class imbalance is a silent problem in object detection.
#      If 'person' has ~8,500 instances and 'stop sign' has ~210,
#      the imbalance ratio is ~40:1. Visualizing this before
#      training helps confirm that YOLOv8's built-in mechanisms
#      (mosaic augmentation, copy-paste, focal loss weighting)
#      are sufficient to handle the imbalance without manual
#      oversampling or loss reweighting.
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

def count_class_distribution(split_name):
    """Count bounding box annotation instances per class for a given split."""
    lbl_dir = f"{BASE_DIR}/labels/{split_name}"
    counts  = defaultdict(int)
    for lbl_file in os.listdir(lbl_dir):
        with open(os.path.join(lbl_dir, lbl_file), 'r') as f:
            for line in f:
                line = line.strip()
                if line:
                    cls_id = int(line.split()[0])
                    counts[cls_id] += 1
    return counts

train_counts = count_class_distribution('train')
val_counts   = count_class_distribution('val')

# Plot side-by-side bar charts ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("Class Distribution Analysis — 15-Class COCO Subset",
             fontsize=14, fontweight='bold')

for ax, counts, split in zip(axes, [train_counts, val_counts], ['Train', 'Val']):
    sorted_items = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    ids    = [SELECTED_CLASSES[i] for i, _ in sorted_items]
    vals   = [v for _, v in sorted_items]
    colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(ids)))
    bars   = ax.barh(ids, vals, color=colors)
    ax.set_xlabel("Number of Instances")
    ax.set_title(f"{split} Split — Class Distribution")
    ax.bar_label(bars, fmt='%d', padding=3, fontsize=8)
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig("/content/class_distribution.png", dpi=150, bbox_inches='tight')
plt.show()
print("Class distribution chart saved to /content/class_distribution.png")

# Imbalance ratio analysis ───────────────────────────────────────────────────
all_counts = list(train_counts.values())
max_count  = max(all_counts)
min_count  = min(all_counts)
ratio      = max_count / max(min_count, 1)

print(f"\nMost common class  : {SELECTED_CLASSES[max(train_counts, key=train_counts.get)]}  "
      f"({max_count:,} instances)")
print(f"Rarest class       : {SELECTED_CLASSES[min(train_counts, key=train_counts.get)]}  "
      f"({min_count:,} instances)")
print(f"Imbalance ratio    : {ratio:.1f}x")

if ratio > 10:
    print("High imbalance detected. YOLOv8 handles this via mosaic augmentation,")
    print("copy-paste augmentation, and focal loss weighting (enabled by default).")
else:
    print("Class balance is acceptable for training.")

In [ ]:
# ============================================================
# CELL 8: Create the dataset configuration YAML file
#
# WHY: YOLOv8 reads this YAML to locate images, labels, and
#      class names. This is the single source of truth for
#      the entire dataset configuration. Without a correct
#      YAML, training cannot start.
#
#      Fields:
#        path  — root directory of the dataset
#        train — relative path to training images
#        val   — relative path to validation images
#        test  — relative path to test images
#        nc    — number of classes (15)
#        names — list of class names in YOLO index order
# ============================================================

import yaml

dataset_config = {
    'path'  : BASE_DIR,
    'train' : 'images/train',
    'val'   : 'images/val',
    'test'  : 'images/test',
    'nc'    : len(SELECTED_CLASSES),     # 15
    'names' : SELECTED_CLASSES
}

yaml_path = f"{BASE_DIR}/dataset.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False, sort_keys=False)

print("dataset.yaml created at:", yaml_path)
print("\nContents:")
print("-" * 40)
with open(yaml_path, 'r') as f:
    print(f.read())
print("-" * 40)

In [ ]:
# ============================================================
# CELL 9: Resolve PyTorch / Ultralytics version compatibility
#
# WHY: Version mismatches between PyTorch and Ultralytics can
#      cause silent failures or incorrect training behaviour.
#      Pinning both libraries ensures the pipeline produces
#      identical results across sessions and environments.
#
#      ultralytics==8.2.0  →  matches the version used in this
#                              project and referenced in the report
#      torch==2.5.1        →  validated against CUDA 12.x drivers
#
# NOTE: A runtime restart is NOT required when running cells
#       sequentially in a fresh session. If you have already
#       imported torch in this session, restart the runtime
#       after running this cell and then re-run from Cell 1.
# ============================================================

!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 -q
!pip install ultralytics==8.2.0 -q

import torch
import ultralytics

print("=" * 50)
print(f"PyTorch version    : {torch.__version__}")
print(f"Ultralytics version: {ultralytics.__version__}")
print(f"CUDA available     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}             : {torch.cuda.get_device_name(i)}")
print("=" * 50)
print("Version compatibility confirmed. Proceeding to training.")

In [ ]:
# ============================================================
# CELL 10: Load YOLOv8s pretrained weights and launch training
#
# MODEL: YOLOv8s (small variant)
#   ~11.2 million trainable parameters  |  ~22 MB model file
#   Pre-trained on full COCO 80-class dataset (yolov8s.pt)
#   Fine-tuned here on the custom 15-class subset
#
# TRAINING CONFIGURATION (as per project report):
#   epochs    = 50     Full training run
#   imgsz     = 640    Input resolution (pixels)
#   batch     = 8      GPU memory-optimised batch size
#   optimizer = AdamW  (Ultralytics default)
#   lr0       = 0.01   Initial LR with cosine annealing decay
#   seed      = 42     Fixed for reproducibility
#
# DEFAULT AUGMENTATIONS (enabled automatically by YOLOv8):
#   Mosaic augmentation    (p=1.0)  —  key for class imbalance
#   Random horizontal flip (p=0.5)
#   HSV color jitter
#   Scale jitter (0.5–1.5x)
#   Translation jitter
#   Copy-paste augmentation
#
# Best checkpoint saved automatically at minimum val loss.
# Output: /content/runs/yolov8s_coco15/weights/best.pt
# ============================================================

from ultralytics import YOLO

# Load YOLOv8s with COCO-pretrained weights
model = YOLO('yolov8s.pt')

total_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
print(f"Model              : YOLOv8s")
print(f"Trainable params   : {total_params:,}")
print(f"Weights loaded from: yolov8s.pt (COCO 80-class pretrained)")
print("\nStarting fine-tuning on custom 15-class dataset...")
print("=" * 55)

results = model.train(
    data      = f"{BASE_DIR}/dataset.yaml",
    epochs    = 50,
    imgsz     = 640,
    batch     = 8,
    optimizer = 'AdamW',
    lr0       = 0.01,
    project   = '/content/runs',
    name      = 'yolov8s_coco15',
    exist_ok  = True,
    seed      = 42,
    verbose   = True
)

print("=" * 55)
print("Training complete.")
print("Best weights saved to: /content/runs/yolov8s_coco15/weights/best.pt")

In [ ]:
# ============================================================
# CELL 11: Locate best checkpoint (best.pt)
#
# WHY A SEPARATE CELL:
#   Separating model loading from evaluation makes it easy to
#   re-run evaluation with different settings (e.g. different
#   conf threshold) without re-running training.
#   Uses glob with getctime ordering to find the most recently
#   saved best.pt even if the run folder name differs.
# ============================================================

import glob
import os
from ultralytics import YOLO

# Search all subfolders of /content/runs/ for best.pt ────────────────────────
weights_list = glob.glob('/content/runs/**/weights/best.pt', recursive=True)

if not weights_list:
    raise FileNotFoundError(
        "best.pt not found. Ensure Cell 10 (training) completed successfully."
    )

# Select the most recently created weights file
best_weights = max(weights_list, key=os.path.getctime)
print(f"Found {len(weights_list)} weights file(s).")
print(f"Loading best weights : {best_weights}")

best_model = YOLO(best_weights)

total_params = sum(p.numel() for p in best_model.model.parameters() if p.requires_grad)
print(f"Model               : YOLOv8s")
print(f"Trainable params    : {total_params:,}")
print("Best model loaded and ready for evaluation.")

In [ ]:
# ============================================================
# CELL 12: Evaluate trained model on the held-out test split
#
# WHY THE TEST SPLIT:
#   The test split was never seen during training or val-based
#   checkpoint selection, providing an unbiased performance
#   estimate that reflects real-world deployment behaviour.
#
# EVALUATION PROTOCOL:
#   Standard COCO detection evaluation
#   conf = 0.25   Confidence threshold
#   iou  = 0.45   NMS IoU threshold
#
# METRICS COMPUTED:
#   mAP@50      IoU ≥ 0.50  (lenient localization)
#   mAP@50-95   Avg over IoU 0.50–0.95, step 0.05
#   Precision   TP / (TP + FP)
#   Recall      TP / (TP + FN)
#   F1 Score    Harmonic mean of Precision and Recall
#
# Project target thresholds (from report Section 1.2):
#   mAP@50 ≥ 0.85  |  Precision ≥ 0.90  |  Recall ≥ 0.85
# ============================================================

metrics = best_model.val(
    data    = f"{BASE_DIR}/dataset.yaml",
    split   = 'test',
    imgsz   = 640,
    conf    = 0.25,
    iou     = 0.45,
    verbose = True
)

# Extract metrics and compute F1 ─────────────────────────────────────────────
map50   = metrics.box.map50
map5095 = metrics.box.map
prec    = metrics.box.mp
rec     = metrics.box.mr
f1      = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

print("\n" + "=" * 55)
print("MODEL EVALUATION RESULTS — TEST SPLIT")
print("=" * 55)
print(f"  mAP@50       : {map50:.4f}   (target ≥ 0.85)")
print(f"  mAP@50-95    : {map5095:.4f}   (target ≥ 0.60)")
print(f"  Precision    : {prec:.4f}   (target ≥ 0.90)")
print(f"  Recall       : {rec:.4f}   (target ≥ 0.85)")
print(f"  F1 Score     : {f1:.4f}   (target ≥ 0.88)")
print("=" * 55)

if map50 >= 0.85:
    print(f"mAP@50 target PASSED ({map50:.4f} >= 0.85)")
else:
    print(f"mAP@50 target NOT MET ({map50:.4f} < 0.85)")

In [ ]:
# ============================================================
# CELL 13: Visualize annotated bounding box predictions
#
# WHY: Quantitative metrics alone do not reveal spatial
#      detection quality. Rendering predicted bounding boxes
#      on sample test images gives qualitative confirmation
#      that the model correctly localizes and classifies all
#      15 object categories.
#
# SETTINGS:
#   conf   = 0.25   Only show boxes with confidence ≥ 25%
#   6 randomly selected images from the test split (seed=42)
#   results[0].plot() renders RGB overlay with class labels
#   Output: /content/sample_predictions.png
# ============================================================

import glob
import random
import matplotlib.pyplot as plt
import os

# Sample 6 random test images ────────────────────────────────────────────────
test_imgs = (glob.glob(f"{BASE_DIR}/images/test/*.jpg")  +
             glob.glob(f"{BASE_DIR}/images/test/*.jpeg") +
             glob.glob(f"{BASE_DIR}/images/test/*.png"))

if not test_imgs:
    raise FileNotFoundError(
        "No test images found. Ensure Cells 5 and 6 ran successfully."
    )

random.seed(42)
num_samples = min(6, len(test_imgs))
sample_imgs = random.sample(test_imgs, num_samples)
print(f"Visualizing {num_samples} sample predictions (conf >= 0.25)...\n")

# Render 2x3 prediction grid ─────────────────────────────────────────────────
cols = 3
rows = (num_samples + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(18, 5 * rows))
fig.suptitle(
    "Sample Annotated Prediction Outputs (Confidence Threshold >= 0.25)",
    fontsize=13, fontweight='bold'
)

ax_flat = axes.flatten() if rows > 1 else axes

for i, img_path in enumerate(sample_imgs):
    results       = best_model.predict(source=img_path, imgsz=640, conf=0.25, verbose=False)
    annotated_rgb = results[0].plot()[:, :, ::-1]   # BGR -> RGB

    ax_flat[i].imshow(annotated_rgb)
    ax_flat[i].axis('off')
    ax_flat[i].set_title(os.path.basename(img_path), fontsize=9)

# Hide unused axes
for j in range(num_samples, rows * cols):
    ax_flat[j].set_visible(False)

plt.tight_layout()
plt.savefig('/content/sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Prediction visualization saved to /content/sample_predictions.png")

In [ ]:
# ============================================================
# CELL 14: Export trained model to ONNX format
#
# WHY ONNX:
#   ONNX (Open Neural Network Exchange) is a cross-platform
#   model interchange format enabling deployment outside of
#   PyTorch / Ultralytics on multiple inference runtimes:
#     OpenCV DNN    : cv2.dnn.readNetFromONNX()
#     ONNX Runtime  : pip install onnxruntime-gpu
#     NVIDIA TensorRT : accelerated edge deployment
#
# EXPORT SETTINGS (as per project report Section 6.2):
#   format  = 'onnx'
#   imgsz   = 640     Fixed input resolution (matches training)
#   dynamic = False   Static input shape (TensorRT compatible)
#   opset   = 17      Compatible with all major ONNX runtimes
#
# Output: best.onnx saved alongside best.pt in weights folder
# ============================================================

import glob
import os
from ultralytics import YOLO

# Locate best.pt ─────────────────────────────────────────────────────────────
weights_list = glob.glob('/content/runs/**/weights/best.pt', recursive=True)

if not weights_list:
    raise FileNotFoundError(
        "No trained weights found. Run Cell 10 (training) before exporting."
    )

best_weights = max(weights_list, key=os.path.getctime)
print(f"Exporting weights  : {best_weights}")

model = YOLO(best_weights)

# Export to ONNX ─────────────────────────────────────────────────────────────
print("Exporting to ONNX (imgsz=640, dynamic=False, opset=17)...")
onnx_path = model.export(
    format  = 'onnx',
    imgsz   = 640,
    dynamic = False,
    opset   = 17
)

# File size report ────────────────────────────────────────────────────────────
pt_size   = os.path.getsize(best_weights)    / 1e6
onnx_size = os.path.getsize(str(onnx_path))  / 1e6

print("\n" + "=" * 55)
print("EXPORT SUCCESSFUL")
print("=" * 55)
print(f"  PyTorch weights : {best_weights}")
print(f"  PyTorch size    : {pt_size:.1f} MB")
print(f"  ONNX model      : {onnx_path}")
print(f"  ONNX size       : {onnx_size:.1f} MB")
print(f"  Opset version   : 17")
print(f"  Input shape     : [1, 3, 640, 640]  (static)")
print("=" * 55)
print("Compatible runtimes: OpenCV DNN | ONNX Runtime | NVIDIA TensorRT")